# BirdCLEF 2026 — Submission Notebook
**Modello:** EfficientNetB0 + transfer learning  
**Task:** Per ogni finestra audio da 5 secondi, predire la probabilità di presenza per 234 specie  
**Vincoli:** Solo CPU, max 90 minuti di runtime

## Cella 1 — Import delle librerie

In [ ]:
import os
# os = modulo per navigare nel filesystem (leggere cartelle, costruire path)

import numpy as np
# numpy = libreria per array numerici veloci
# es: np.array([1,2,3]) crea un array, np.zeros((128,320)) crea matrice di zeri

import pandas as pd
# pandas = libreria per tabelle (DataFrame)
# useremo pd.DataFrame() per costruire il CSV di submission

import librosa
# librosa = libreria specializzata per audio
# librosa.load() legge file .ogg/.wav e restituisce (segnale, sample_rate)
# librosa.feature.melspectrogram() converte il segnale in mel-spettrogramma

import tensorflow as tf
# tensorflow = framework di deep learning
# useremo tf.keras.models.load_model() per caricare il tuo best_model.keras

import warnings
warnings.filterwarnings('ignore')
# ignora i warning per tenere l'output pulito

print('Librerie caricate con successo!')
print('TensorFlow versione:', tf.__version__)

## Cella 2 — Configurazione dei path e parametri

In [ ]:
# ============================================================
# PATH DEL MODELLO
# Su Kaggle, i tuoi dataset caricati si trovano in /kaggle/input/
# Il path è: /kaggle/input/NOME-DEL-TUO-DATASET/best_model.keras
# SOSTITUISCI 'birdclef2026-model' con il nome che hai dato al tuo dataset!
# ============================================================
MODEL_PATH = '/kaggle/input/birdclef2026-model/best_model.keras'

# PATH DEI DATI DI TEST DELLA COMPETITION
# Kaggle mette automaticamente i dati della competition in /kaggle/input/birdclef-2026/
TEST_AUDIO_DIR = '/kaggle/input/birdclef-2026/test_soundscapes'

# PATH DEL FILE DI OUTPUT
# Il notebook deve scrivere qui il file di submission
SUBMISSION_PATH = '/kaggle/working/submission.csv'

# ============================================================
# PARAMETRI AUDIO — devono essere IDENTICI a quelli usati in training!
# ============================================================
SAMPLE_RATE = 32000      # frequenza di campionamento in Hz (32000 campioni al secondo)
DURATION = 5.0           # ogni finestra audio dura 5 secondi
N_MELS = 128             # numero di bande mel nello spettrogramma (asse Y)
HOP_LENGTH = 512         # quanti campioni si 'salta' tra una colonna e l'altra dello spettrogramma
N_FFT = 1024             # dimensione della finestra FFT (Fast Fourier Transform)
FMIN = 20                # frequenza minima considerata (Hz)
FMAX = 16000             # frequenza massima considerata (Hz)

# DIMENSIONE INPUT DEL MODELLO
# Dal model.summary() sappiamo che EfficientNetB0 si aspetta (128, 320, 3)
# 128 = N_MELS (bande mel)
# 320 = frame temporali (5 sec * 32000 Hz / 512 hop_length ≈ 313, arrotondiamo a 320)
# 3   = canali RGB (EfficientNet è addestrato su immagini a colori)
TARGET_HEIGHT = 128      # altezza spettrogramma in pixel (= N_MELS)
TARGET_WIDTH = 320       # larghezza spettrogramma in pixel (frame temporali)

print('Configurazione caricata!')
print(f'  Modello: {MODEL_PATH}')
print(f'  Audio test: {TEST_AUDIO_DIR}')
print(f'  Output: {SUBMISSION_PATH}')

## Cella 3 — Lista delle 234 specie (colonne del submission.csv)

In [ ]:
# La competition fornisce un file 'sample_submission.csv' che ci dice:
# 1. Il formato esatto del file di submission
# 2. L'ORDINE ESATTO delle 234 specie (colonne)
# Questo ordine è FONDAMENTALE: se sbagliamo l'ordine, le predizioni sono sbagliate!

SAMPLE_SUB_PATH = '/kaggle/input/birdclef-2026/sample_submission.csv'

# Leggiamo il sample submission
# pd.read_csv() legge un file CSV e lo trasforma in un DataFrame (tabella)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

# Mostriamo le prime righe per capire la struttura
print('Prime 3 righe del sample_submission:')
print(sample_sub.head(3))
print()
print(f'Forma del sample_submission: {sample_sub.shape}')
# shape = (numero righe, numero colonne)

# Estraiamo i nomi delle specie: sono tutte le colonne tranne la prima ('row_id')
# sample_sub.columns è la lista di tutti i nomi delle colonne
# [1:] significa 'prendi tutto tranne il primo elemento (indice 0)'
SPECIES_LIST = list(sample_sub.columns[1:])

print(f'Numero di specie: {len(SPECIES_LIST)}')
print(f'Prime 5 specie: {SPECIES_LIST[:5]}')

## Cella 4 — Caricamento del modello

In [ ]:
print('Caricamento del modello...')

# tf.keras.models.load_model() carica il modello salvato in precedenza
# compile=False significa che non ricompiliamo l'ottimizzatore (non serve per la predizione)
# Questo risparmia un po' di tempo e memoria
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

print('Modello caricato!')
print(f'  Input shape atteso: {model.input_shape}')
# Dovrebbe stampare: (None, 128, 320, 3)
# None = batch size (variabile)
# 128 = altezza spettrogramma
# 320 = larghezza spettrogramma  
# 3   = canali RGB

print(f'  Output shape: {model.output_shape}')
# Dovrebbe stampare: (None, 234) — una probabilità per ogni specie

## Cella 5 — Funzione: audio → mel-spettrogramma

In [ ]:
def audio_to_spectrogram(audio_array, sr=SAMPLE_RATE):
    """
    Converte un array audio in un mel-spettrogramma pronto per il modello.
    
    Input:
        audio_array: array numpy con il segnale audio (forma: (N,) dove N = numero campioni)
        sr: sample rate (default 32000 Hz)
    
    Output:
        spettrogramma normalizzato, forma (TARGET_HEIGHT, TARGET_WIDTH, 3)
        cioè (128, 320, 3) — pronto per EfficientNet
    """
    
    # STEP 1: Calcola il mel-spettrogramma
    # librosa.feature.melspectrogram() applica la Trasformata di Fourier a breve termine
    # e poi converte le frequenze nella scala mel (più simile alla percezione umana)
    mel_spec = librosa.feature.melspectrogram(
        y=audio_array,      # y = segnale audio (array 1D)
        sr=sr,              # sr = sample rate
        n_mels=N_MELS,      # numero di bande mel (128)
        hop_length=HOP_LENGTH,  # passo tra finestre successive (512 campioni)
        n_fft=N_FFT,        # dimensione della finestra FFT (1024)
        fmin=FMIN,          # frequenza minima (20 Hz)
        fmax=FMAX           # frequenza massima (16000 Hz)
    )
    # mel_spec ha forma (128, T) dove T = numero frame temporali
    
    # STEP 2: Converti in scala logaritmica (decibel)
    # I suoni naturali hanno un range dinamico enorme.
    # La scala logaritmica comprime questo range e migliora l'apprendimento.
    # librosa.power_to_db() fa: 10 * log10(mel_spec)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    # Ora i valori sono in decibel, tipicamente tra -80 e 0
    
    # STEP 3: Ridimensiona alla larghezza esatta che si aspetta il modello (320 frame)
    # Il numero di frame T dipende dalla lunghezza esatta dell'audio.
    # Se T < 320: aggiungiamo zeri a destra (padding)
    # Se T > 320: tagliamo a 320
    current_width = mel_spec_db.shape[1]  # quanti frame abbiamo ora
    
    if current_width < TARGET_WIDTH:
        # np.pad aggiunge padding
        # ((0,0), (0, TARGET_WIDTH - current_width)) significa:
        # - asse 0 (righe/mel): non aggiungere nulla
        # - asse 1 (colonne/tempo): aggiungi zeri a destra
        # mode='constant' = riempi con un valore costante (default: 0)
        mel_spec_db = np.pad(
            mel_spec_db,
            ((0, 0), (0, TARGET_WIDTH - current_width)),
            mode='constant'
        )
    elif current_width > TARGET_WIDTH:
        # [:, :TARGET_WIDTH] = prendi tutte le righe, ma solo le prime TARGET_WIDTH colonne
        mel_spec_db = mel_spec_db[:, :TARGET_WIDTH]
    
    # Ora mel_spec_db ha forma esattamente (128, 320)
    
    # STEP 4: Normalizzazione [0, 1]
    # I valori in dB vanno da circa -80 a 0.
    # Normalizziamo in [0, 1] sottraendo il minimo e dividendo per il range.
    # Questo è lo stesso che hai fatto in training!
    min_val = mel_spec_db.min()  # valore minimo dello spettrogramma
    max_val = mel_spec_db.max()  # valore massimo
    
    if max_val > min_val:  # evita divisione per zero
        mel_spec_norm = (mel_spec_db - min_val) / (max_val - min_val)
    else:
        mel_spec_norm = np.zeros_like(mel_spec_db)  # array di tutti zeri
    
    # STEP 5: Converti da (128, 320) a (128, 320, 3)
    # EfficientNet si aspetta immagini RGB (3 canali).
    # Duplichiamo il canale grigio 3 volte.
    # np.stack([a, a, a], axis=-1) impila 3 copie lungo l'ultimo asse
    # axis=-1 significa 'aggiungi un asse alla fine'
    mel_spec_rgb = np.stack([mel_spec_norm, mel_spec_norm, mel_spec_norm], axis=-1)
    # Ora ha forma (128, 320, 3)
    
    return mel_spec_rgb.astype(np.float32)  # float32 è più efficiente in memoria di float64


print('Funzione audio_to_spectrogram definita!')
print('Esempio: un audio da 5 sec a 32000 Hz → spettrogramma (128, 320, 3)')

## Cella 6 — Funzione: processa un intero file audio (diviso in finestre da 5s)

In [ ]:
def process_audio_file(audio_path):
    """
    Legge un file audio, lo divide in finestre da 5 secondi,
    e restituisce una lista di (row_id, spettrogramma) per ogni finestra.
    
    BirdCLEF 2026 usa il formato row_id = 'NOMEFILE_SECONDI'
    Esempio: 'soundscape_001_5' = file soundscape_001, finestra che finisce al secondo 5
    
    Input:
        audio_path: path completo del file .ogg o .wav
    
    Output:
        lista di tuple (row_id, spettrogramma)
    """
    
    # Estrai il nome del file senza estensione
    # os.path.basename() prende solo il nome del file (senza la cartella)
    # os.path.splitext() separa nome e estensione: ('soundscape_001', '.ogg')
    filename = os.path.splitext(os.path.basename(audio_path))[0]
    # Esempio: '/kaggle/input/.../soundscape_001.ogg' → 'soundscape_001'
    
    results = []  # lista dove metteremo i risultati
    
    try:
        # Carica l'intero file audio
        # librosa.load() restituisce:
        #   y = array 1D con il segnale audio
        #   sr = sample rate effettivo
        # sr=SAMPLE_RATE forza il resampling a 32000 Hz se necessario
        # mono=True converte in mono (un solo canale) se stereo
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        # y ha forma (N,) dove N = durata_secondi * 32000
        # Esempio: 60 secondi → y ha 1.920.000 elementi
        
        # Calcola quanti campioni corrispondono a 5 secondi
        # Esempio: 5 * 32000 = 160.000 campioni
        samples_per_window = int(DURATION * SAMPLE_RATE)  # = 160000
        
        # Calcola il numero totale di finestre intere da 5 secondi
        # len(y) = numero totale di campioni
        # // = divisione intera (arrotonda verso il basso)
        # Esempio: 1.920.000 // 160.000 = 12 finestre
        n_windows = len(y) // samples_per_window
        
        # Iteriamo su ogni finestra
        for i in range(n_windows):
            
            # Calcola l'indice di inizio e fine della finestra
            start = i * samples_per_window
            # Esempio: finestra 0 → start=0, finestra 1 → start=160000, ecc.
            end = start + samples_per_window
            
            # Estrai la finestra dal segnale audio
            # y[start:end] = 'prendi l'array y dall'indice start all'indice end'
            window = y[start:end]
            
            # Converti la finestra in spettrogramma
            spec = audio_to_spectrogram(window, sr=sr)
            # spec ha forma (128, 320, 3)
            
            # Costruisci il row_id nel formato BirdCLEF
            # Il row_id indica 'quale file, a quale secondo'
            # Il secondo è la FINE della finestra: finestra 0 finisce al secondo 5
            end_second = (i + 1) * int(DURATION)  # 5, 10, 15, 20, ...
            row_id = f'{filename}_{end_second}'
            # Esempio: 'soundscape_001_5', 'soundscape_001_10', ...
            
            results.append((row_id, spec))
        
    except Exception as e:
        # Se c'è un errore (file corrotto, formato strano, ecc.),
        # stampiamo il messaggio ma non blocchiamo tutto il notebook
        print(f'  ERRORE nel file {filename}: {e}')
    
    return results


print('Funzione process_audio_file definita!')

## Cella 7 — Loop principale: predizione su tutti i file di test

In [ ]:
# Trova tutti i file audio nella cartella di test
# os.listdir() restituisce una lista di tutti i file nella cartella
all_files = os.listdir(TEST_AUDIO_DIR)

# Filtra solo i file .ogg (formato usato da BirdCLEF 2026)
# f.endswith('.ogg') = True se il file finisce con '.ogg'
audio_files = [f for f in all_files if f.endswith('.ogg')]
audio_files.sort()  # ordina alfabeticamente per riproducibilità

print(f'Trovati {len(audio_files)} file audio di test')

# Liste dove accumuleremo i risultati
all_row_ids = []      # lista di row_id (stringhe)
all_predictions = []  # lista di array di predizioni (234 valori ciascuno)

# BATCH SIZE per la predizione
# Invece di fare la predizione una finestra alla volta (lento),
# accumuliamo finestre in 'batch' e le processiamo insieme.
# Batch size = quante finestre processiamo insieme.
# Su CPU, 16-32 è un buon compromesso velocità/memoria.
BATCH_SIZE = 16

batch_specs = []  # spettrogrammi nel batch corrente
batch_ids = []    # row_id nel batch corrente

def flush_batch():
    """Processa il batch corrente e salva le predizioni."""
    if not batch_specs:
        return
    
    # np.array(batch_specs) converte la lista di array in un unico array 4D
    # Se batch_specs ha 16 elementi di forma (128, 320, 3),
    # il risultato ha forma (16, 128, 320, 3)
    X = np.array(batch_specs, dtype=np.float32)
    
    # model.predict() fa la predizione su tutto il batch
    # verbose=0 = non stampare la barra di progresso (riduce l'output)
    # preds ha forma (16, 234): per ogni delle 16 finestre, 234 probabilità
    preds = model.predict(X, verbose=0)
    
    # Aggiungiamo i risultati alle liste principali
    all_row_ids.extend(batch_ids)
    all_predictions.extend(preds)
    # extend() aggiunge tutti gli elementi di una lista a un'altra
    
    # Svuota i batch per il prossimo ciclo
    batch_specs.clear()
    batch_ids.clear()


# === LOOP PRINCIPALE ===
for file_idx, filename in enumerate(audio_files):
    
    # Costruisci il path completo del file
    # os.path.join() unisce cartella e nome file con il separatore corretto
    # Esempio: '/kaggle/input/birdclef-2026/test_soundscapes' + 'file.ogg'
    #        → '/kaggle/input/birdclef-2026/test_soundscapes/file.ogg'
    audio_path = os.path.join(TEST_AUDIO_DIR, filename)
    
    # Stampa progresso ogni 10 file
    if file_idx % 10 == 0:
        print(f'Processing file {file_idx+1}/{len(audio_files)}: {filename}')
    
    # Processa il file (divide in finestre da 5s e crea spettrogrammi)
    windows = process_audio_file(audio_path)
    
    # Aggiungi ogni finestra al batch corrente
    for row_id, spec in windows:
        batch_ids.append(row_id)
        batch_specs.append(spec)
        
        # Quando il batch è pieno, esegui la predizione
        if len(batch_specs) >= BATCH_SIZE:
            flush_batch()

# Processa le finestre rimaste nell'ultimo batch (potrebbe essere < BATCH_SIZE)
flush_batch()

print(f'\nPredizioni completate!')
print(f'Totale finestre processate: {len(all_row_ids)}')

## Cella 8 — Costruisci e salva il submission.csv

In [ ]:
# Converti la lista di predizioni in un array numpy 2D
# Se all_predictions ha N elementi di forma (234,),
# np.array(all_predictions) dà forma (N, 234)
predictions_array = np.array(all_predictions)

print(f'Shape delle predizioni: {predictions_array.shape}')
# Dovrebbe stampare: (numero_finestre, 234)

# Crea il DataFrame di submission
# pd.DataFrame(data, columns=colonne) crea una tabella con i dati specificati
# predictions_array ha una riga per ogni finestra e 234 colonne (una per specie)
submission_df = pd.DataFrame(
    predictions_array,
    columns=SPECIES_LIST  # i nomi delle colonne sono le 234 specie nell'ordine giusto
)

# Aggiungi la colonna 'row_id' come PRIMA colonna
# insert(0, 'row_id', all_row_ids) inserisce alla posizione 0 (inizio)
submission_df.insert(0, 'row_id', all_row_ids)

# Mostra le prime righe per verifica
print('\nPrime 3 righe della submission:')
print(submission_df.head(3))
print(f'\nForma della submission: {submission_df.shape}')
# Dovrebbe essere (numero_finestre, 235) = righe x (1 row_id + 234 specie)

# Salva il file CSV
# index=False = non scrivere l'indice (0,1,2,...) come colonna aggiuntiva
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f'\nSubmission salvata in: {SUBMISSION_PATH}')
print('Ora puoi premere "Submit" su Kaggle!')

## Cella 9 — Verifica finale (opzionale ma consigliata)
Questa cella controlla che il file sia nel formato corretto prima di submittare.

In [ ]:
# Rileggi il file appena scritto per verificare che sia corretto
check_df = pd.read_csv(SUBMISSION_PATH)

print('=== VERIFICA SUBMISSION ===')
print(f'Numero righe: {len(check_df)}')
print(f'Numero colonne: {len(check_df.columns)}')
print(f'Prima colonna: {check_df.columns[0]} (deve essere "row_id")')
print(f'Numero specie: {len(check_df.columns) - 1} (deve essere 234)')

# Controlla che i valori siano tra 0 e 1 (probabilità valide)
# .values restituisce i valori come array numpy (senza la colonna row_id)
pred_values = check_df.iloc[:, 1:].values  
# iloc[:, 1:] = prendi tutte le righe, colonne dall'indice 1 in poi

print(f'Valore minimo predizioni: {pred_values.min():.4f} (deve essere >= 0)')
print(f'Valore massimo predizioni: {pred_values.max():.4f} (deve essere <= 1)')
print(f'Presenza di NaN: {np.isnan(pred_values).any()} (deve essere False)')

# Confronta il formato con il sample_submission
sample_sub_check = pd.read_csv(SAMPLE_SUB_PATH)
print(f'\nColonne corrispondono al sample_submission: {list(check_df.columns) == list(sample_sub_check.columns)}')
# Deve essere True!

print('\n✅ Submission pronta per essere caricata su Kaggle!')